In [27]:
# en el modulo Qiskit Optimization hay una implementacion del algoritmo Grover Adaptive Search

from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.algorithms import GroverOptimizer

from qiskit.primitives import StatevectorSampler
from qiskit_aer import AerSimulator

from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver

In [10]:
# definimos el problema

qp = QuadraticProgram()

qp.binary_var('x')
qp.binary_var('y')

qp.minimize(linear={'x':2,'y':2}, quadratic={('x','y'):-3})
print(qp.prettyprint())

Problem name: 

Minimize
  -3*x*y + 2*x + 2*y

Subject to
  No constraints

  Binary variables (2)
    x y



In [ ]:
sampler = StatevectorSampler()


grover_optimizer = GroverOptimizer(num_value_qubits=3, num_iterations=2, sampler=sampler)

result = grover_optimizer.solve(qp)
print(result)

fval=0.0, x=0.0, y=0.0, status=SUCCESS


In [ ]:
# queremos minimizar 
qp1 = QuadraticProgram()

qp1.binary_var('x')
qp1.binary_var('y')
qp1.binary_var('z')

qp1.minimize(linear={'x':3,'y':2,'z':-3}, quadratic={('x','y'):3})

sampler = StatevectorSampler()

# necesitamos como num_value_qubits un numero de qubits suficiente para almacenar 
# en representacion complemento a 2 todas las posibles
# salidas de la funcion a minimizar 
grover_optimizer1 = GroverOptimizer(num_value_qubits=5, num_iterations=3, sampler=sampler)

result = grover_optimizer1.solve(qp1)
print(result)

fval=-3.0, x=0.0, y=0.0, z=1.0, status=SUCCESS


In [100]:
# problemas con restricciones
qp = QuadraticProgram()
qp.binary_var('x')
qp.binary_var('y')
qp.binary_var('z')

qp.minimize(linear={'x':2}, quadratic={('x','z'):1,('z','y'):-2})
qp.linear_constraint(linear={'x':2, 'y':-1, 'z':1}, sense='<=', rhs=2)
print(qp.prettyprint())

Problem name: 

Minimize
  x*z - 2*y*z + 2*x

Subject to
  Linear constraints (1)
    2*x - y + z <= 2  'c0'

  Binary variables (3)
    x y z



In [101]:
# podriamos seguir el mismo procedimiento que antes y crear una instancia de GroverOptimizer,
# y usar su metodo solve con qp, pero no sabemos cuantos qubits vamos a necesitar.
# Al ser un problema con restricciones, podemos reescribirlo como QUBO mediante variables de slack
# pero no vamos a saber cuantos qubits necesitamos para almacenar en representacion complemento a 2
# todas las posibles salidas de la funcion a minimizar 

# vamos a reescribir el problema en QUBO y decidir cuantos qubits necesitamos en base a los resultados de eso

from qiskit_optimization.converters import QuadraticProgramToQubo

qp_to_qubo = QuadraticProgramToQubo()
qubo = qp_to_qubo.convert(qp)
print(qubo.prettyprint())

Problem name: 

Minimize
  6*c0@int_slack@0^2 + 24*c0@int_slack@0*c0@int_slack@1 + 24*c0@int_slack@1^2
  + 24*x*c0@int_slack@0 + 48*x*c0@int_slack@1 + 24*x^2 - 24*x*y + 25*x*z
  - 12*y*c0@int_slack@0 - 24*y*c0@int_slack@1 + 6*y^2 - 14*y*z
  + 12*z*c0@int_slack@0 + 24*z*c0@int_slack@1 + 6*z^2 - 24*c0@int_slack@0
  - 48*c0@int_slack@1 - 46*x + 24*y - 24*z + 24

Subject to
  No constraints

  Binary variables (5)
    x y z c0@int_slack@0 c0@int_slack@1



In [111]:
# vamos a necesitar al menos 10 qubits porque
# la suma de todos los coeficientes positivos es 271 y
# la suma de los coeficientes negativos es -216

sampler = StatevectorSampler()

grover_optimizer = GroverOptimizer(num_value_qubits=10, num_iterations=4, sampler=sampler)

result = grover_optimizer.solve(qubo)
print(result)

fval=-2.0, x=0.0, y=1.0, z=1.0, c0@int_slack@0=0.0, c0@int_slack@1=1.0, status=SUCCESS


In [112]:
# si quiero que no aparezcan las variables de slack: 
result = grover_optimizer.solve(qp)
print(result)

fval=-2.0, x=0.0, y=1.0, z=1.0, status=SUCCESS
